# 13 — Two-Stage PCA + Robust Mahalanobis Clustering

Replaces the flat 416 × 550 merged PCA matrix from notebook 04 with a
**two-stage PCA compressed matrix** that retains more original HOG variance
while staying within the Robust Mahalanobis safety limit.

| Stage | Input | Threshold | Output per type |
|-------|-------|-----------|----------------|
| Stage 1 | 8,100 raw HOG features (per image type) | 80% variance | ~200 components |
| Merge | 5 × Stage 1 outputs | — | ~957 columns |
| Stage 2 | merged matrix | 70% variance | **135 final components** |

135 < 416/3 ≈ 138.67 → **SAFE for Robust Mahalanobis**.

Clustering: `kmedoids.fasterpam` on the precomputed Robust Mahalanobis distance matrix.  
Evaluation: spread · silhouette · ARI · NMI — same metrics as notebook 04.

## Install

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

for pkg in ["kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Imports

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

import kmedoids
from robust_mixed_dist.mixed import robust_mahalanobis_dist_matrix, S_robust
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 1 — Load raw HOG features and CDR

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

HOG_LEN     = 8_100
IMAGE_TYPES = ["cor", "gfc_sag", "gfc_tra", "masked_tra", "sbj_sag"]

# ── Raw HOG features ──────────────────────────────────────────────────────────
print("Loading raw_hog_features_per_image.csv ...", end=" ", flush=True)
df_hog = pd.read_csv(os.path.join(NB_DIR, "raw_hog_features_per_image.csv"))
patient_ids = df_hog["patient_id"].values
print(f"done  {df_hog.shape}")

hog_matrices = {}
for itype in IMAGE_TYPES:
    cols = [f"{itype}_HOG_{i}" for i in range(HOG_LEN)]
    hog_matrices[itype] = df_hog[cols].to_numpy(dtype=np.float64)
    print(f"  {itype:<14} {hog_matrices[itype].shape}")

# ── CDR ───────────────────────────────────────────────────────────────────────
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})
df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)
cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"\nPatients          : {len(patient_ids)}")
print(f"Patients with CDR : {mask_cdr.sum()}")

Loading raw_hog_features_per_image.csv ... done  (416, 40501)
  cor            (416, 8100)
  gfc_sag        (416, 8100)
  gfc_tra        (416, 8100)
  masked_tra     (416, 8100)
  sbj_sag        (416, 8100)

Patients          : 416
Patients with CDR : 235


## Step 2 — Two-stage PCA

**Stage 1 (80%):** Compress each image type's 8,100 raw HOG features independently.  
**Stage 2 (70%):** Compress the concatenated Stage 1 outputs into the final matrix.

Target: ≤ 138 final components (safe for Robust Mahalanobis with 416 patients).

In [4]:
S1_THRESH = 0.80   # Stage 1 variance threshold
S2_THRESH = 0.70   # Stage 2 variance threshold

# ── Stage 1: compress each image type independently ───────────────────────────
print(f"Stage 1 PCA ({S1_THRESH*100:.0f}% variance per image type):")
stage1_blocks = []
for itype, X_hog in hog_matrices.items():
    pca1  = PCA(n_components=S1_THRESH, random_state=42)
    X_s1  = pca1.fit_transform(X_hog)
    stage1_blocks.append(X_s1)
    var1  = pca1.explained_variance_ratio_.sum() * 100
    print(f"  {itype:<14} {X_hog.shape} → {X_s1.shape}  ({var1:.1f}% of raw HOG variance)")

# ── Merge Stage 1 outputs ──────────────────────────────────────────────────────
X_merged = np.concatenate(stage1_blocks, axis=1)
print(f"\nMerged Stage 1 matrix : {X_merged.shape}")

# ── Stage 2: compress merged matrix ───────────────────────────────────────────
print(f"\nStage 2 PCA ({S2_THRESH*100:.0f}% variance on merged matrix):")
pca2    = PCA(n_components=S2_THRESH, random_state=42)
X_final = pca2.fit_transform(X_merged)
var2    = pca2.explained_variance_ratio_.sum() * 100

N_FINAL        = X_final.shape[1]
SAFE_THRESHOLD = len(patient_ids) / 3

print(f"  {X_merged.shape} → {X_final.shape}  ({var2:.1f}% of merged variance)")
print(f"\nFinal matrix X_final  : {X_final.shape}")
print(f"Safe limit            : < {SAFE_THRESHOLD:.2f}  (= 416 / 3)")
print(f"Status                : {'SAFE ✓' if N_FINAL < SAFE_THRESHOLD else 'UNSAFE ✗'}  ({N_FINAL} components)")

Stage 1 PCA (80% variance per image type):
  cor            (416, 8100) → (416, 212)  (80.0% of raw HOG variance)
  gfc_sag        (416, 8100) → (416, 199)  (80.0% of raw HOG variance)
  gfc_tra        (416, 8100) → (416, 202)  (80.1% of raw HOG variance)
  masked_tra     (416, 8100) → (416, 200)  (80.1% of raw HOG variance)
  sbj_sag        (416, 8100) → (416, 144)  (80.1% of raw HOG variance)

Merged Stage 1 matrix : (416, 957)

Stage 2 PCA (70% variance on merged matrix):
  (416, 957) → (416, 135)  (70.2% of merged variance)

Final matrix X_final  : (416, 135)
Safe limit            : < 138.67  (= 416 / 3)
Status                : SAFE ✓  (135 components)


## Step 3 — Robust Mahalanobis distance matrix

Computed once on `X_final` and reused for both k=3 and k=4.

In [5]:
print(f"Computing Robust Mahalanobis on X_final {X_final.shape} ...", end=" ", flush=True)
S_est = S_robust(X_final, method="trimmed", alpha=0.05)
D     = robust_mahalanobis_dist_matrix(X_final, S_est)
print(f"done")
print(f"Distance matrix D : {D.shape}")
print(f"Value range       : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric      : {np.allclose(D, D.T)}")
print(f"NaN values        : {np.isnan(D).sum()}")

Computing Robust Mahalanobis on X_final (416, 135) ... done
Distance matrix D : (416, 416)
Value range       : [0.0000, 28.9967]
Is symmetric      : True
NaN values        : 0


## Step 4 — Cluster and evaluate (k=3 and k=4)

Same evaluation as notebook 04: spread · silhouette · ARI · NMI.

In [6]:
all_results = []

print(f"{'='*62}")
print(f"  Two-stage PCA (S1={S1_THRESH*100:.0f}%, S2={S2_THRESH*100:.0f}%)  |  "
      f"X_final={X_final.shape}  |  CDR patients: {mask_cdr.sum()}")
print(f"{'='*62}")

for k in [3, 4]:
    result = kmedoids.fasterpam(D, medoids=k, random_state=42)
    labels = np.array(result.labels, dtype=int)

    # ── % healthy per cluster ──────────────────────────────────────────────────
    df_eval  = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
    df_eval  = df_eval[df_eval["CDR"].notnull()].copy()
    pct_tab  = pd.crosstab(
        df_eval["cluster"], df_eval["CDR"], normalize="index"
    ).mul(100).round(1)
    pct_h   = pct_tab[0.0] if 0.0 in pct_tab.columns \
              else pd.Series(0.0, index=pct_tab.index)
    spread  = float(pct_h.max() - pct_h.min())

    # ── CDR crosstab ───────────────────────────────────────────────────────────
    df_ev2 = df_eval.copy()
    df_ev2["CDR"] = df_ev2["CDR"].astype(str)
    ct  = pd.crosstab(df_ev2["cluster"], df_ev2["CDR"],
                      margins=True, margins_name="Total")
    print(f"\n  k={k}  loss={result.loss:.4f}  "
          f"sizes={ {c: int((labels==c).sum()) for c in sorted(set(labels))} }")
    print("  Count:")
    print(ct.to_string())
    print("  Row %:")
    print(pct_tab.to_string())

    # ── Metrics ────────────────────────────────────────────────────────────────
    sil = silhouette_score(D, labels, metric="precomputed")
    ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
    nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

    print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

    all_results.append({
        "method":             f"two_stage_pca_s1={S1_THRESH*100:.0f}_s2={S2_THRESH*100:.0f}",
        "n_final_components": N_FINAL,
        "k":                  k,
        "silhouette":         round(sil, 4),
        "ari":                round(ari, 4),
        "nmi":                round(nmi, 4),
        "spread_pct_healthy": round(spread, 2),
    })

print("\n\nClustering complete.")

  Two-stage PCA (S1=80%, S2=70%)  |  X_final=(416, 135)  |  CDR patients: 235

  k=3  loss=5948.0069  sizes={np.int64(0): 106, np.int64(1): 100, np.int64(2): 210}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         36   18    4    1     59
1         33   18    4    0     55
2         66   34   20    1    121
Total    135   70   28    2    235
  Row %:
CDR       0.0   0.5   1.0  2.0
cluster                       
0        61.0  30.5   6.8  1.7
1        60.0  32.7   7.3  0.0
2        54.5  28.1  16.5  0.8

  spread=6.50%  sil=-0.0014  ARI=-0.0196  NMI=0.0140

  k=4  loss=5909.9986  sizes={np.int64(0): 55, np.int64(1): 90, np.int64(2): 185, np.int64(3): 86}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         16   11    3    0     30
1         32   15    3    1     51
2         59   29   19    1    108
3         28   15    3    0     46
Total    135   70   28    2    235
  Row %:
CDR       0.0   0.5   1.0  2.0
cluste

## Step 5 — Summary and save

In [7]:
df_results = pd.DataFrame(all_results)

print(f"Two-stage PCA + Robust Mahalanobis  "
      f"(Stage1={S1_THRESH*100:.0f}%, Stage2={S2_THRESH*100:.0f}%, final={N_FINAL} components)")
print("=" * 70)
print(df_results.to_string(index=False))
print(f"\nSilhouette: all {len(patient_ids)} patients")
print(f"ARI / NMI : {mask_cdr.sum()} patients with CDR  (CDR × 2)")

out_path = os.path.join(NB_PADDED, "results_two_stage_pca_mahalanobis.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Two-stage PCA + Robust Mahalanobis  (Stage1=80%, Stage2=70%, final=135 components)
                   method  n_final_components  k  silhouette     ari    nmi  spread_pct_healthy
two_stage_pca_s1=80_s2=70                 135  3     -0.0014 -0.0196 0.0140                 6.5
two_stage_pca_s1=80_s2=70                 135  4     -0.0050 -0.0175 0.0175                 9.4

Silhouette: all 416 patients
ARI / NMI : 235 patients with CDR  (CDR × 2)

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_two_stage_pca_mahalanobis.csv
